In [ ]:
import random
import numpy as np
from datasets import load_dataset, get_dataset_config_names
import time

# ============================================================================
# 📚 학습 목표: 한국어 질의응답(QA) 데이터셋을 활용한 AI 프롬프트 생성 실습
# 📝 데이터셋명: ziozzang/Korean_QA_gen_datasets
# 🔍 데이터셋의 의미: AI 모델이 생성한 고품질의 한국어 질문(Question)과 정답(Answer) 쌍 데이터입니다.
# ✨ 이 실습으로 배울 것: 실제 LLM(거대언어모델)을 구동하기 전에, 데이터셋의 구조를 분석하고,
#   모델이 가장 잘 학습할 수 있는 '질문-답변' 프롬프트 템플릿을 데이터 기반으로 설계하는 방법을 배웁니다.
# ============================================================================

# --- 설정값 ---
DATASET_NAME = "ziozzang/Korean_QA_gen_datasets"
SAMPLE_COUNT = 5 # 초보자에게 보여주기 위해 샘플 개수를 줄였습니다.

# --- 1. 데이터셋 로딩 준비 (가장 중요한 단계!) ---

# 💡 튜터 코멘트: 데이터 로딩은 가장 까다로운 부분입니다. 큰 데이터셋을 다룰 때는 메모리 관리를 위해 '스트리밍' 기능을 사용하는 것이 좋습니다.
# 하지만 환경에 따라 스트리밍이 실패할 수 있으니, 오류 처리(try-except)를 반드시 거쳐야 합니다!

dataset = None
print("✨ 🚀 데이터셋 로딩을 시도합니다... 잠시만 기다려주세요!")

try:
    # 🟢 1차 시도: 스트리밍 모드로 데이터셋을 로드 (메모리 효율성 최고!)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 축하합니다! 스트리밍 모드로 데이터셋을 성공적으로 로드했습니다. 👏 (메모리 절약 모드)")

except Exception as e:
    # 🔴 2차 시도: 스트리밍에 실패했을 경우 (네트워크 또는 환경 문제)
    print(f"⚠️ 스트리밍 로드 실패 오류 발생: {e}. 일반 다운로드 방식으로 전환합니다.")
    try:
        # 테스트용으로 매우 작은 부분만 다운로드하여 로드를 시도합니다.
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("✅ 대체 로딩 성공! 소량의 데이터셋으로 안전하게 진행하겠습니다. 👍")
    except Exception as e_fallback:
        print(f"😭 심각한 에러가 발생했습니다. 데이터셋 로드에 실패했습니다: {e_fallback}")
        exit()


# --- 2. 데이터 샘플링 (전체 데이터를 사용하면 시간이 너무 오래 걸려요!) ---

# 💡 튜터 코멘트: 751개의 데이터를 다 볼 필요는 없어요! 우선 대표적인 5개만 뽑아서 실습을 진행해 봅시다.
# 'take()' 메소드를 사용하여 상위 K개만 효율적으로 가져올게요! (메모리 절약의 마법 ✨)

# 데이터셋이 스트리밍 객체인지 확인하여 샘플링 패턴을 적용합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print(f"\n🧠 상위 {SAMPLE_COUNT}개의 샘플만 뽑아 분석할 준비를 합니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    
    # 스트리밍 데이터는 리스트로 변환하기 어려우므로, 순회하며 리스트에 담습니다.
    sample_data_list = []
    for i, sample in enumerate(sampled_dataset_iterator):
        sample_data_list.append(sample)
else:
    # 일반 데이터셋 (Dataset)
    sample_data_list = list(dataset.take(SAMPLE_COUNT))

# --- 3. 초보자용 실습 1: 데이터 구조 및 통계 분석 (데이터 탐험가 모드!) ---

print("\n" + "="*70)
print("📊 실습 1: 샘플 데이터를 통한 QA 통계 분석 (데이터 탐험가 모드)")
print("="*70)

question_lengths = []
answer_lengths = []
sample_count = len(sample_data_list)

print(f"✨ 분석할 샘플 개수: {sample_count}개")

# 🧑‍💻 루프를 돌면서 데이터를 하나씩 확인하고, 질문과 답변의 길이를 세어봅니다.
for i, sample in enumerate(sample_data_list):
    question = sample['question']
    answer = sample['answer']
    
    # 📏 질문과 답변의 길이를 리스트에 저장합니다.
    question_lengths.append(len(question))
    answer_lengths.append(len(answer))

# 📈 분석 결과 출력
avg_q_len = np.mean(question_lengths)
std_q_len = np.std(question_lengths)
avg_a_len = np.mean(answer_lengths)
std_a_len = np.std(answer_lengths)

print("-" * 30)
print("📊 [질문 길이 분석] (Question Length)")
print(f"   평균 길이: {avg_q_len:.2f} 글자")
print(f"   표준편차: {std_q_len:.2f} 글자 (질문 길이가 얼마나 들쑥날쑥한지 보여줘요)")

print("\n📊 [답변 길이 분석] (Answer Length)")
print(f"   평균 길이: {avg_a_len:.2f} 글자")
print(f"   표준편차: {std_a_len:.2f} 글자")
print("-" * 70)

# --- 4. 초보자용 실습 2: AI 프롬프트 템플릿 생성 (만능 LLM 엔지니어 모드!) ---

print("\n" + "="*70)
print("🤖 실습 2: LLM 학습용 프롬프트 템플릿 생성 실습")
print("="*70)

# 💡 튜터 코멘트: LLM은 데이터가 어떤 '형식'으로 들어오는지에 민감합니다.
# 단순한 질문/답변보다, "다음 질문에 대해 이렇게 대답해줘"라는 형태로 묶어주는 것이 가장 좋습니다.
# 이것이 바로 프롬프트 엔지니어링의 핵심이에요!

print("\n✨ 목표: '질문(Q)'과 '정답(A)'을 결합하여, 모델이 학습하기 좋은 대화 형식의 프롬프트를 만듭니다.")

# 첫 번째 샘플을 골라 예시를 보여줍니다.
if sample_data_list:
    sample_example = sample_data_list[0]
    
    Q = sample_example['question']
    A = sample_example['answer']

    print("\n🌟 [첫 번째 샘플 기반 프롬프트 예시]")
    print("---------------------------------------------------")
    print("<< System Prompt Start >>")
    print("당신은 한국어 QA 전문가입니다. 주어진 질문에 대해 핵심적인 정보를 제공하는 답변을 해줘.")
    print("<< User Input >>")
    print(f"Q: {Q}")
    print("<< Model Output >>")
    print(f"A: {A}")
    print("---------------------------------------------------")
    
    # 🧑‍💻 모든 샘플에 대해 이 템플릿을 반복 적용하는 시뮬레이션
    print("\n✅ 💡 총 샘플에 대해 반복적으로 실행하면, 다음과 같은 형태로 대규모 학습 데이터(Train Data)를 구축하게 됩니다.")
    print("\n(이런 식으로 모든 샘플을 반복하여, 모델이 패턴을 익히도록 학습 데이터셋을 구성하는 것이 목표입니다!)")

else:
    print("❌ 분석할 샘플이 없습니다. 로딩 단계를 다시 확인해 주세요.")

print("\n✨ 축하합니다! 🎉 데이터 로딩부터 분석, 실제 AI 활용까지 하나의 스크립트로 경험해 봤습니다!")
print("파이썬 코딩 실력, 최고예요! 👍")